# Predicting late delivery of consignments
Question: can `late_delivery` be predicted at dispatch time?

Design decisions (from checklist.md and columns.csv):
- `consignment` is an identifier -> dropped (each row is a unique consignment, so no entity grouping is needed).
- `claim_paid_eur` is recorded after delivery, when a claim is settled (it is ~79 EUR for late vs ~1 EUR for on-time rows). It is not known at prediction time -> dropped as leakage.
- The model is meant for future consignments, so the test set is the **latest rows** by `dispatched_on`, and CV inside training uses forward-chaining splits.

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.base import clone
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, brier_score_loss
df=pd.read_csv('data.csv',parse_dates=['dispatched_on']).sort_values('dispatched_on').reset_index(drop=True)
df['hour']=df.dispatched_on.dt.hour; df['dow']=df.dispatched_on.dt.dayofweek
print(df.shape, df.late_delivery.mean())

## 1. Test split: latest 20% of rows by dispatch time. Not touched until step 6.

In [ ]:
cut=int(len(df)*0.8)
train,test=df.iloc[:cut].copy(),df.iloc[cut:].copy()
print(train.dispatched_on.max(), test.dispatched_on.min())
# drift check on TRAIN only
print(train.groupby(train.dispatched_on.dt.to_period('M')).late_delivery.mean().round(3).to_string())

The late rate in training varies a lot over time, so a calendar/month feature would not extrapolate; none is used. Only dispatch-time features: carrier, distance_km, weight_kg, parcels, hour, day of week.

## 2-4. Candidates, named before any scoring
1. `baseline`: predicts training prevalence (trivial).
2. `logreg`: standardised numeric + one-hot carrier, logistic regression (C in {0.1, 1, 10}).
3. `hgb`: HistGradientBoosting (max_depth in {2, 4}; learning rate 0.05, 200 iterations fixed).

Primary metric: average precision (PR-AUC) on out-of-fold predictions; ROC AUC secondary. Decision threshold: maximise F1 on pooled out-of-fold predictions. Preprocessing lives inside the Pipeline so it is fit within each fold.

In [ ]:
num=['distance_km','weight_kg','parcels','hour','dow']; cat=['carrier']; feats=num+cat
def pre(): return ColumnTransformer([('n',StandardScaler(),num),('c',OneHotEncoder(handle_unknown='ignore'),cat)])
cands={'baseline':Pipeline([('m',DummyClassifier(strategy='prior'))])}
for C_ in [0.1,1,10]: cands[f'logreg_C{C_}']=Pipeline([('p',pre()),('m',LogisticRegression(C=C_,max_iter=1000))])
for d in [2,4]: cands[f'hgb_d{d}']=Pipeline([('p',pre()),('m',HistGradientBoostingClassifier(max_depth=d,learning_rate=0.05,max_iter=200,random_state=0))])
print(list(cands))

## 5. Forward-chaining CV on training rows only

In [ ]:
X,y=train[feats],train.late_delivery.values
tscv=TimeSeriesSplit(n_splits=5)
oof={};res=[]
for name,mod in cands.items():
    p=np.full(len(train),np.nan); aps=[];aucs=[]
    for tr,va in tscv.split(X):
        m=clone(mod).fit(X.iloc[tr],y[tr]); p[va]=m.predict_proba(X.iloc[va])[:,1]
        aps.append(average_precision_score(y[va],p[va])); aucs.append(roc_auc_score(y[va],p[va]))
    oof[name]=p; res.append((name,np.mean(aps),np.std(aps),np.mean(aucs)))
cv=pd.DataFrame(res,columns=['model','AP_mean','AP_sd','AUC_mean']).set_index('model'); print(cv.round(3))

In [ ]:
best=cv.drop('baseline').AP_mean.idxmax()
mask=~np.isnan(oof[best]); ths=np.linspace(0.02,0.8,79)
f1s=[f1_score(y[mask],oof[best][mask]>=t) for t in ths]
thr=float(ths[int(np.argmax(f1s))])
print(best,'threshold',thr,'CV F1',max(f1s))

## Choice (recorded before test scoring)
The chosen model is whichever candidate has the best CV average precision, as printed above, with the F1-maximising CV threshold. If it does not clearly beat the baseline in CV, the honest conclusion is that late delivery is not predictable from these features. Nothing is changed after seeing the test set.

## 6. Refit on all training rows, score test once

In [ ]:
final=clone(cands[best]).fit(X,y)
pt=final.predict_proba(test[feats])[:,1]; yt=test.late_delivery.values
base=clone(cands['baseline']).fit(X,y).predict_proba(test[feats])[:,1]
out=pd.DataFrame({'chosen':[roc_auc_score(yt,pt),average_precision_score(yt,pt),brier_score_loss(yt,pt),f1_score(yt,pt>=thr)],
                  'baseline':[0.5,yt.mean(),brier_score_loss(yt,base),f1_score(yt,base>=thr)]},index=['ROC AUC','Avg precision','Brier','F1 @ CV threshold'])
print('test rows',len(test),'test late rate',yt.mean().round(4),'train late rate',y.mean().round(4)); print(out.round(3))

## Conclusion
See the single test score above (headline). Caveats: the late rate shifts strongly over time (it falls to about 1% in the latest months), so test prevalence differs from training and threshold-based metrics (F1) and calibration (Brier) are affected by that shift, while ranking metrics are more robust. With so few positives in the test period, estimates are noisy.